# 03 - Modelo, avaliação, diagnóstico e previsão 2027
Ridge sobre o desvio relativo à média de cada origem, dentro de Pipeline (imputação, escala e one-hot ajustados só no treino).

Regra de escolha definida antes de olhar o teste: métrica principal = MAE do total mensal; a validação e o backtest de origem móvel decidem; o teste é usado uma única vez.

In [1]:
import sys, warnings
from pathlib import Path
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
from src.config import *
from src.dados import *
from src.variaveis import *
np.random.seed(SEMENTE)
PROCESSADO.mkdir(parents=True, exist_ok=True)
FIG = RAIZ / "docs" / "figuras"; FIG.mkdir(parents=True, exist_ok=True)
import json
from src.particao import particionar_temporal
from src.baseline import *
from src.modelo import ModeloRelativo, ajustar_modelo, prever
from src.backtest import backtest, resumo_backtest
from src.diagnostico import residuos, resumo_por, autocorrelacao_total_mensal, maiores_residuos

In [2]:
bruto = carregar_bruto()
assert bruto is not None, "Coloque a planilha em dados/bruto/ (veja dados/bruto/LEIAME.md)"
bruto = bruto.drop(columns="arquivo_origem")
base, log, total_entrada = preparar_base_mensal(bruto)
feats, _ = criar_variaveis(base)
treino, valid, teste = particionar_temporal(feats)
ALFA = 30.0  # fixo: o CV de 2 dobras não enxerga efeito de mês (cada mês aparece pela 1a vez); 10 a 100 dão resultado quase igual no backtest

## 1. Validação cruzada temporal (só no treino)

In [3]:
busca = ajustar_modelo(treino)
print("score por alfa (neg MAE):", dict(zip(busca.cv_results_["param_alpha"], busca.cv_results_["mean_test_score"].round(2))))
print("Todos iguais: as dobras validam meses que o treino ainda não viu, então o efeito do mês não entra. Alfa fixado em", ALFA)

score por alfa (neg MAE): {np.float64(1.0): np.float64(-49.63), np.float64(3.0): np.float64(-49.63), np.float64(10.0): np.float64(-49.63), np.float64(30.0): np.float64(-49.63), np.float64(100.0): np.float64(-49.63)}
Todos iguais: as dobras validam meses que o treino ainda não viu, então o efeito do mês não entra. Alfa fixado em 30.0


## 2. Comparação na validação (mesma partição, mesma métrica)

In [4]:
mod = ModeloRelativo(alpha=ALFA).fit(treino, treino[COL_ALVO])
prev_v = {"baseline trivial (média por origem)": prever_media_treino(treino, valid),
          "baseline operacional (último valor)": prever_ultimo_valor(treino, valid),
          "modelo (Ridge relativo)": mod.predict(valid)}
tab_v = tabela_metricas(valid, prev_v).round(3); display(tab_v)

,abordagem,MAE do total mensal (principal),WAPE por série (secundária),viés do total mensal
0,baseline trivial (média por origem),140.583,0.169,16.917
1,baseline operacional (último valor),263.667,0.272,263.667
2,modelo (Ridge relativo),140.583,0.169,16.917


## 3. Backtest de origem móvel (vários cortes, horizonte 1 a 3 meses)

In [5]:
bt = backtest(feats, min_treino=6, h_max=3, alpha=ALFA)
res_bt = resumo_backtest(bt).round(3); display(res_bt)
print("Por horizonte (MAE do total):"); display(bt.assign(a=bt.erro_total.abs()).pivot_table(index="abordagem", columns="horizonte", values="a", aggfunc="mean").round(1))

,MAE do total mensal,WAPE por origem,viés do total (previsto - real),n previsões
abordagem,,,,
Ridge relativo,145.622,0.164,-60.215,21
mediana por origem,146.286,0.174,-51.190,21
média por origem,150.368,0.165,-47.705,21
último valor,231.048,0.237,-6.857,21


Por horizonte (MAE do total):


horizonte,1,2,3
abordagem,,,
Ridge relativo,148.5,150.5,136.0
mediana por origem,141.3,153.6,144.3
média por origem,152.7,156.5,140.1
último valor,223.5,250.1,218.8


## 4. Teste (uma única vez): treino + validação -> teste

In [6]:
tv = pd.concat([treino, valid])
mod_t = ModeloRelativo(alpha=ALFA).fit(tv, tv[COL_ALVO])
prev_t = {"baseline trivial (média por origem)": prever_media_treino(tv, teste),
          "baseline operacional (último valor)": prever_ultimo_valor(tv, teste),
          "modelo (Ridge relativo)": mod_t.predict(teste)}
tab_t = tabela_metricas(teste, prev_t).round(3); display(tab_t)

,abordagem,MAE do total mensal (principal),WAPE por série (secundária),viés do total mensal
0,baseline trivial (média por origem),109.121,0.155,-52.030
1,baseline operacional (último valor),283.667,0.190,-283.667
2,modelo (Ridge relativo),103.057,0.152,-77.327


## 5. Diagnóstico de resíduos (modelo, no teste)

In [7]:
res = residuos(teste, prev_t["modelo (Ridge relativo)"])
display(resumo_por(res, COL_ORIGEM).round(1))
display(resumo_por(res, COL_MES).round(1))
print("Autocorrelação lag 1 dos resíduos do total:", autocorrelacao_total_mensal(res), "(só 3 meses: não calculável)")
display(maiores_residuos(res, 5).round(1))
fig, ax = plt.subplots(figsize=(7,3.2))
r = res.groupby(COL_ORIGEM)["residuo"].mean().sort_values()
ax.barh(r.index, r.values, color="#0B3954"); ax.axvline(0, color="k", lw=.8); ax.set_title("Resíduo médio por origem no teste (real - previsto)")
fig.tight_layout(); fig.savefig(FIG/"residuos_origem.png", dpi=150)

,n,residuo_medio,residuo_abs_medio
origem,,,
MAURILIO DE ALMEIDA,3,80.2,80.2
PARTICULARES,3,11.9,11.9
ROSEANE DORE,3,34.7,40.9
SAO LUCAS,3,15.8,29.1
SUS,3,-66.0,66.0
VIVER MAIS,3,0.7,5.1


,n,residuo_medio,residuo_abs_medio
mes,,,
2026-07-01,6,39.6,46.1
2026-08-01,6,-6.4,41.4
2026-09-01,6,5.5,29.1


Autocorrelação lag 1 dos resíduos do total: None (só 3 meses: não calculável)


,origem,mes,exames,previsto,residuo
11,MAURILIO DE ALMEIDA,2026-07-01,593,471.2,121.8
68,SUS,2026-08-01,510,624.2,-114.2
12,MAURILIO DE ALMEIDA,2026-08-01,543,456.2,86.8
39,ROSEANE DORE,2026-07-01,344,267.9,76.1
69,SUS,2026-09-01,568,632.1,-64.1


## 6. Previsão de agosto a dezembro de 2027
Modelo reajustado com os 14 meses. Faixa indicativa: erro típico do total no backtest (percentis 10 e 90 dos erros observados). Horizonte de 11 a 15 meses **nunca foi validado**.

In [8]:
mod_f = ModeloRelativo(alpha=ALFA).fit(feats, feats[COL_ALVO])
fut = meses_futuros(base, MESES_PREVISAO)
fut["previsto"] = mod_f.predict(fut)
media_hist = feats.groupby(COL_ORIGEM)[COL_ALVO].mean()
fut["media_historica"] = fut[COL_ORIGEM].map(media_hist)
por_origem = fut.pivot(index=COL_ORIGEM, columns=COL_MES, values="previsto").round(0)
por_origem.columns = [c.strftime("%b/%Y") for c in por_origem.columns]
tot = fut.groupby(COL_MES)["previsto"].sum()
q10, q90 = bt["erro_total"].quantile([0.1, 0.9]).values
faixa = pd.DataFrame({"previsto_total": tot.round(0), "faixa_baixa": (tot - q90).round(0), "faixa_alta": (tot - q10).round(0),
                      "vs_capacidade_1850": (tot - CAPACIDADE_MEDIA).round(0)})
display(por_origem); display(faixa)
fut.to_csv(PROCESSADO/"previsao_2027_por_origem.csv", index=False); faixa.to_csv(PROCESSADO/"previsao_2027_total.csv")
fig, ax = plt.subplots(figsize=(9,3.4))
hist = feats.groupby(COL_MES)[COL_ALVO].sum()
ax.plot(hist.index, hist.values, marker="o", color="#0B3954", label="real (total)")
ax.plot(tot.index, tot.values, marker="o", color="#1479A8", ls="--", label="previsto 2027")
ax.fill_between(tot.index, (tot-q90).values, (tot-q10).values, color="#1479A8", alpha=.15, label="faixa indicativa (backtest)")
ax.axhline(CAPACIDADE_MEDIA, color="#C86A12", ls=":", label="capacidade informada (1.850)")
ax.legend(fontsize=7); ax.set_title("Total mensal: histórico e previsão"); fig.tight_layout(); fig.savefig(FIG/"previsao_2027.png", dpi=150)

,Aug/2027,Sep/2027,Oct/2027,Nov/2027,Dec/2027
origem,,,,,
MAURILIO DE ALMEIDA,487.0,494.0,533.0,486.0,465.0
PARTICULARES,6.0,6.0,7.0,6.0,6.0
ROSEANE DORE,275.0,278.0,300.0,274.0,262.0
SAO LUCAS,88.0,89.0,96.0,88.0,84.0
SUS,629.0,637.0,688.0,628.0,600.0
VIVER MAIS,11.0,11.0,12.0,11.0,10.0


,previsto_total,faixa_baixa,faixa_alta,vs_capacidade_1850
mes,,,,
2027-08-01,1495.0,1251.0,1772.0,-355.0
2027-09-01,1515.0,1271.0,1792.0,-335.0
2027-10-01,1635.0,1391.0,1912.0,-215.0
2027-11-01,1493.0,1249.0,1770.0,-357.0
2027-12-01,1427.0,1183.0,1704.0,-423.0


In [9]:
saida = {"validacao": tab_v.to_dict("records"), "teste": tab_t.to_dict("records"), "backtest": res_bt.reset_index().to_dict("records"),
         "previsao_total": faixa.reset_index().astype(str).to_dict("records"), "alfa": ALFA,
         "faixa_q10_q90": [float(q10), float(q90)], "n_treino": len(treino), "n_valid": len(valid), "n_teste": len(teste)}
(PROCESSADO/"resultados.json").write_text(json.dumps(saida, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
print("resultados salvos em dados/processado/")

resultados salvos em dados/processado/
